In [1]:
import numpy as np
import warnings

from sklearn.metrics.cluster import adjusted_rand_score
from tqdm import tqdm

from CANO import CANO
from Graph_Atom.ADMM import ADMM_cls
from Graph_Atom.graph_func_ADOP import GraphFunc_ADOP
from load_atom import load_atom
from utils.algorithm_utils_orig import *
from utils.SBM import SBM
from utils.utils import *

warnings.filterwarnings("ignore", category=UserWarning, message="Graph is not fully connected, spectral embedding may not work as expected.")


In [2]:
def claulate_grad_coefs_ADOP(atoms, coefs, graph, grad_sig):
    grad_coefs = []
    for atom, coef in zip(atoms, coefs):

        tmp0 = graph.X @ np.linalg.inv(atom.R) @ np.transpose(graph.X)
        tmp1 = (-1 / (coef**2)) * np.sum(np.dot(atom.e**2, tmp0))
        tmp2 = np.sum(atom.e**2) * np.trace(atom.R)
        grad1 = 2 * config.mu1 * (tmp1 + tmp2)

        tmp3 = np.einsum("i, ij-> j", atom.e**2, grad_sig.pi)
        grad2 = -2 * config.mu2 * np.sum(np.dot(tmp3, atom.b))

        grad3 = config.mu0

        grad = grad1 + grad2 + grad3
        grad_coefs.append(grad)
    return np.array(grad_coefs)


def update_coefs_ADOP(coefs, grad_coefs, lr):
    new_coefs = Coefs()
    for i, coef in enumerate(coefs):
        coef -= lr * grad_coefs[i]
        new_coefs.add_coef(coef)
    return new_coefs



In [3]:
##### Utility funcitons
def create_random_normalized_vector(n):
    vector = np.random.randn(n)
    normalized_vector = vector / np.linalg.norm(vector)
    return normalized_vector

def get_random_coef_between_1_and_(n):
    return np.random.uniform(1, n)

def matrix_dot_product(A, B):
    return np.sum(A * B)

# def invert_matrix(matrix):
#     try:
#         return np.linalg.inv(matrix)
#     except np.linalg.LinAlgError:
#         return np.eye(matrix.shape[0])

In [4]:
def initialize_atoms(graph, init="rand", init_atoms_e=None, init_atoms_R=None, init_atoms_b=None, init_coefs=None):

    atoms_e_opt = []
    atoms_R_opt = []
    atoms_b_opt = []
    coeffss_opt = []

    for i in range(config.num_clusters):
        atoms_e_opt.append(graph.E[:, i]) 
        atoms_R_opt.append(graph.Ri_matrices[i])
        atoms_b_opt.append(graph.b[i])
        coeffss_opt.append(graph.λ[i])

    if init == "rand":
        atoms_e = []
        atoms_R = []
        atoms_b = []
        coeffss = []
        for i in range(config.num_clusters):
            atoms_e.append(create_random_normalized_vector(config.num_nodes)) 
            atoms_R.append(create_Ri_symmetric_matrix_with_constrained_eigenvalues(config.dimension, config.rho_min, config.rho_max, noise_level=0.))
            atoms_b.append(create_multiple_noisy_probability_vectors(1, config.num_clusters, noise_level=1)[0])
            coeffss.append(get_random_coef_between_1_and_(config.num_nodes))

    elif init == "opt":
        atoms_e = atoms_e_opt
        atoms_R = atoms_R_opt
        atoms_b = atoms_b_opt
        coeffss = coeffss_opt

    elif init == "zeros":
        atoms_e = []
        atoms_R = []
        atoms_b = []
        coeffss = []
        for i in range(config.num_clusters):
            atoms_e.append(np.zeros(config.num_nodes)) 
            atoms_R.append(np.zeros((config.dimension, config.dimension)))
            atoms_b.append(np.zeros(config.num_clusters))
            coeffss.append(0)

    elif init == "load":
        atoms_e = init_atoms_e 
        atoms_R = init_atoms_R 
        atoms_b = init_atoms_b 
        coeffss = init_coefs

    return atoms_e, atoms_R, atoms_b, coeffss, atoms_e_opt, atoms_R_opt, atoms_b_opt, coeffss_opt

In [5]:
def calculate_sig_from_atoms(atoms_e, atoms_R, atoms_b, coefs):
    sig_L = np.zeros((config.num_nodes, config.num_nodes))
    sig_Rv_s = np.zeros((config.num_nodes, config.dimension, config.dimension))
    sig_Piv_s = np.zeros((config.num_nodes, config.num_clusters))

    for e, R, b, coef in zip(atoms_e, atoms_R, atoms_b, coefs):
        sig_L += coef * np.outer(e, e)
        sig_Rv_s += coef * np.einsum("i,jk->ijk", (e**2), R)
        sig_Piv_s += coef * np.einsum("i,j->ij",  (e**2), b)

    return sig_L, sig_Rv_s, sig_Piv_s

def compute_term_0(graph, sig_L):
    if config.Abar:
        term_0 = -matrix_dot_product(graph.Abar, sig_L) / config.num_nodes
    else:
        r = graph.A - sig_L
        term_0 = 0.5 * matrix_dot_product(r, r) / config.num_nodes
    return term_0


def compute_term_1(graph, sig_Rv_s):
    if config.mu1 != 0:
        sig_Rv_inv = np.array([np.inalg.inv(matrix) for matrix in sig_Rv_s])
        term_1 = np.einsum("ijk,ijk->", graph.cov_ADOP, sig_Rv_inv) / config.num_features
        term_11 = np.trace(np.sum(sig_Rv_s, axis=0)) / config.num_features
    else:
        term_1 = 0
        term_11 = 0
    return term_1 + term_11


def compute_term_2(graph, sig_Piv_s):
    if config.mu2 != 0:
        train_labels = graph.y[graph.train_mask]
        train_nodes = np.arange(graph.num_nodes)[graph.train_mask]
        term_2 = np.sum(sig_Piv_s[train_nodes, train_labels]) / config.num_clusters
    else:
        term_2 = 0
    return term_2

def compute_term_3(coefs):
    return np.sum(coefs) / config.num_clusters



def compute_objective_value(graph, atoms_e, atoms_R, atoms_b, coefs):
    sig_L, sig_Rv_s, sig_Piv_s = calculate_sig_from_atoms(atoms_e, atoms_R, atoms_b, coefs)

    term_0 = compute_term_0(graph, sig_L)
    term_1 = compute_term_1(graph, sig_Rv_s)
    term_2 = compute_term_2(graph, sig_Piv_s)
    term_3 = compute_term_3(coefs)

    objective = term_0 + config.mu1 * term_1 + config.mu2 * term_2 + config.mu0 * term_3
    
    return objective



In [9]:
def ADOP(graph):

    ## Initialization
    atoms_e, atoms_R, atoms_b, coefs, atoms_e_opt, atoms_R_opt, atoms_b_opt, coefs_opt = initialize_atoms(graph, init=config.initialize_atoms)

    f_sig = compute_objective_value(graph, atoms_e, atoms_R, atoms_b, coefs)
    f_opt = compute_objective_value(graph, atoms_e_opt, atoms_R_opt, atoms_b_opt, coefs_opt)

    f_values = [f_sig]
    results = []
    accuracies = []

    # bar = tqdm(total=config.total_iterations, leave=False)
    # for outer_iter in range(config.total_iterations):

    #     #### do alternating minimization
    #     for itr in range(config.local_iter_ALM):

            # grad = f.calc_grad_ADOP(sig)
            # grad_coefs = claulate_grad_coefs_ADOP(atoms, coefs, graph, grad)

            # old_atoms = deepcopy(atoms)
            # for atom, coef in zip(atoms, coefs):
            #     atom.update_atom_ADOP(grad, config.lr, coef, graph)

            # ### based on what we had before
            # grad_without_L = deepcopy(grad)
            # grad_without_L.L = np.zeros_like(grad.L)
            # grad_coefs = np.array(
            #     [grad_without_L.dot(atom.calc_sig_term()) for atom in old_atoms]
            # )


            # coefs = update_coefs_ADOP(coefs, grad_coefs, config.lr)
    #         sig = atoms.calc_sig(coefs)
    #         sig.L = L_current - config.lr * grad.L

    #     ##### do alternating projection
    #     for itr in range(config.local_iter_ALP):
    #         #### project to L=Aλ
    #         A = np.array(
    #             [np.outer(atom.e, atom.e) for atom in atoms]
    #         )  # shape = (num_atoms, n, n)
    #         B = np.einsum("ijk,ljk->il", A, A)  # shape = (num_atoms, num_atoms)
    #         y_tmp = np.einsum("kij,k->ij", A, coefs) - sig.L  # shape = (n, n)
    #         y_bar = np.array([np.einsum("ij,ij->", A[i], y_tmp) for i in range(c)])
    #         tmp = np.linalg.inv(B + np.eye(num_atoms))
    #         alpha = np.matmul(tmp, y_bar)  # shape = (num_atoms)
    #         lagrange_z = y_tmp - np.einsum("kij,k->ij", A, alpha)

    #         sig.L += lagrange_z
    #         coefs -= np.einsum("ijk,jk->i", A, lagrange_z)

    #         #### project coefficients to be non-negative
    #         coefs = np.maximum(coefs, 0)
    #         #### project L to the box
    #         L_current = deepcopy(sig.L)
    #         sig = atoms.calc_sig(Coefs(coefs))
    #         sig.L = project_to_bound(L_current)

    #     f_sig = f(sig, ADMM=False)
    #     f_values.append(f_sig)

    #     # f_sig = f(sig, ADMM=False)

    #     y_pred = np.argmax(sig.pi, axis=1)
    #     y_pred_test = y_pred[graph.test_mask]
    #     if y_pred_test.shape[0] > 0:
    #         acc = calc_accuracy(y_pred_test, graph.y[graph.test_mask])
    #     else:
    #         acc = 1

    #     results = {
    #         f"f": f"{f_sig:,.4f}",
    #         "f_opt": f"{f_opt:,.4f}",
    #         "acc": acc,
    #     }

    #     # if log_results:
    #     bar.set_postfix(results)
    #     bar.update()
    #     f_values.append(f_sig)
    #     accuracies.append(acc)

    return (atoms_e, atoms_R, atoms_b, coefs, atoms_e_opt, atoms_R_opt, atoms_b_opt, coefs_opt, f_opt, f_values)

In [ ]:
graph = SBM(
    config.samples_per_cluster,
    config.p,
    config.q,
    config.dimension,
    config.signal_subspace_dim,
    train_ratio=config.train_ratio,
)

(atoms_e, atoms_R, atoms_b, coefs, atoms_e_opt, atoms_R_opt, atoms_b_opt, coefs_opt, f_opt, f_values ) = ADOP(graph)

# log_results(atoms, coefs, graph, results, save_path)
# plot_results2(f_values, accuracies, atoms, save_path, f_opt)
# save_model_params2(atoms, coefs, graph, save_path)

In [3]:
import numpy as np
def compute_inner_products(tensor):
    return np.einsum('nik,njk->ij', tensor, tensor)


def compute_inner_products(tensor):
  n, d, _ = tensor.shape
  result = np.zeros((n, n))
  for i in range(n):
      for j in range(n):
          result[i, j] = np.inner(tensor[i].flatten(), tensor[j].flatten())
  return result


A = np.arange(12).reshape(3, 2,2)
print(A)
print(compute_inner_products(A))
print(compute_inner_products2(A))

[[[ 0  1]
  [ 2  3]]

 [[ 4  5]
  [ 6  7]]

 [[ 8  9]
  [10 11]]]
[[187 241]
 [241 319]]
[[ 14.  38.  62.]
 [ 38. 126. 214.]
 [ 62. 214. 366.]]
